# Método de Elementos Finitos (FEM)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/maurimendiluce/AN2026/blob/main/elementos_finitos/FEM_1D.ipynb)

## Reacción–difusión: 

Consideremos el problema $$-\varepsilon u'' + u = f$$ con $u(0)=u(1)=0$ la forma débil es $\varepsilon\int_0^1 u'v' + \int_0^1 uv=\int_0^1 fv$, y el sistema queda
$$(\varepsilon K + M)\,\alpha = b,$$
donde $K$ es la matriz de rigidez de siempre y $M_{ij}=\int\varphi_i\varphi_j$ es la **matriz de masa**. 

**Ejemplo:** $f(x)=x$. La solución exacta es $u(x)=x-\dfrac{\sinh(x/\sqrt\varepsilon)}{\sinh(1/\sqrt\varepsilon)}$: sigue a $x$ salvo en una *capa límite* de ancho $\approx\sqrt\varepsilon$ cerca de $x=1$.


In [1]:
using LinearAlgebra, Plots

In [ ]:
u_rd(x, ε) = x - exp((x-1)/√ε) * (1 - exp(-2x/√ε)) / (1 - exp(-2/√ε))
f_rd(x) = x

malla_uniforme(N) = collect(range(0, 1, length=N))      # N nodos

function malla_shishkin(N, ε)
  Ne = N - 1
  @assert iseven(Ne)
  τ = min(0.5, 2*sqrt(ε)*log(Ne))
  return vcat(collect(range(0, 1-τ, length=Ne÷2+1))[1:end-1],
              collect(range(1-τ, 1, length=Ne÷2+1)))
end


### Malla de Shishkin

La capa límite de $-\varepsilon u''+u=f$ en $x=1$ tiene ancho $\sim\sqrt\varepsilon$. Una malla uniforme necesita $h\lesssim\sqrt\varepsilon$ en todo el intervalo, y eso son muchísimos nodos. Como sabemos de antemano **dónde** está la capa, usamos una malla **uniforme a trozos**, con dos zonas:

$$[0,\,1-\tau]\quad\text{(zona gruesa)}\qquad\text{y}\qquad [1-\tau,\,1]\quad\text{(zona fina)},$$

donde el punto de transición es
$$\tau=\min\left\{\tfrac12,\;2\sqrt\varepsilon\,\ln N_e\right\},\qquad N_e=N-1 \text{ (cantidad de elementos)}.$$

Cada zona recibe la mitad de los elementos ($N_e/2$), así que los pasos son
$$H=\frac{2(1-\tau)}{N_e}\quad\text{(zona gruesa)},\qquad h=\frac{2\tau}{N_e}\quad\text{(zona fina)}.$$

**Por qué funciona.**
- $\tau\approx\sqrt\varepsilon\ln N_e$ es un poco más ancho que la capa. En la zona fina el paso es $h\sim\sqrt\varepsilon\,\ln N_e/N_e\ll\sqrt\varepsilon$, así que la capa queda bien resuelta.
- Fuera de la capa la solución es suave (sigue a $x$) y alcanza con pasos grandes.
- Con la misma cantidad de nodos que una malla uniforme, el costo es el mismo. Solo cambia dónde se ubican los nodos.
- El error resulta casi independiente de $\varepsilon$ (convergencia *uniforme en $\varepsilon$*). Con una malla uniforme el error depende fuertemente de $\varepsilon$.

Si $\sqrt\varepsilon\ln N_e\ge\tfrac12$, entonces $\tau=\tfrac12$ y la malla es uniforme (la capa ya no es "fina" respecto de $h$).

**Limitación.** Hay que conocer la posición de la capa antes de armar la malla. Si hubiera capas en los dos bordes, se refinaría en ambos extremos.

In [ ]:
function plot_mallas(N, ε)
  xu = malla_uniforme(N)
  xs = malla_shishkin(N, ε)
  τ  = 1 - xs[N÷2 + 1]            # ancho de la zona refinada [1-τ, 1]

  plt = plot(xlabel="x", yticks=([1, 2], ["Shishkin", "uniforme"]), ylims=(0.5, 2.5),
             title="Mallas con N = $N nodos, ε = $ε  (τ = $(round(τ, sigdigits=3)))",
             legend=false, size=(900, 300))
  scatter!(plt, xu, fill(2, N), marker=:vline, ms=8, c=:red)
  scatter!(plt, xs, fill(1, N), marker=:vline, ms=8, c=:blue)
  vline!(plt, [1-τ], ls=:dash, c=:gray)       # inicio de la capa
  return plt
end

p1 = plot_mallas(21, 1e-4)
p2 = plot_mallas(33, 1e-6)
plot(p1, p2, layout=(2,1), size=(900, 500))